# 05 — Career Path & Mobility
**Goal**: Map career trajectories and identify mobility patterns

**ML Progression**: Statistical → Markov Chain → K-Means Clustering

**HR Value**: Succession planning, internal mobility programs

**Employee Value**: Visible career roadmap and growth opportunities

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

cwd = Path.cwd()
if (cwd / 'data/raw/employee_data.csv').exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / 'data/raw/employee_data.csv').exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError('Cannot find project root')
os.chdir(PROJECT_ROOT)
PROJECT_ROOT = Path.cwd().resolve()

ANALYSIS_DIR = PROJECT_ROOT / 'data/analysis/05_career'
FIGURES_DIR = PROJECT_ROOT / 'reports/figures'
Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(ANALYSIS_DIR / 'dataset.parquet')
print(f'Loaded: {len(df)} employees, {len(df.columns)} cols')

## 1. Tenure & Career Landscape

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df['tenure_years'].dropna(), bins=30)
axes[0].set_title('Tenure Distribution')
df['job_family'].value_counts().plot(kind='bar', ax=axes[1])
axes[1].set_title('Job Family Distribution')
axes[1].tick_params(axis='x', rotation=45)
df.groupby('job_family')['tenure_years'].mean().plot(kind='bar', ax=axes[2])
axes[2].set_title('Avg Tenure by Job Family')
axes[2].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/05_career_landscape.png', bbox_inches='tight')
plt.show()

## 2. Markov Chain (Job Family Transitions)

In [ ]:
# Build transition matrix: job_family -> tenure_bucket -> next state
df['tenure_bin'] = pd.cut(df['tenure_years'], bins=[0, 1, 2, 5, 10, 50], labels=['<1', '1-2', '2-5', '5-10', '10+'])
transition = pd.crosstab(df['job_family'], df['tenure_bin'], normalize='index')
print('Transition Probabilities (Job Family to Tenure Bucket):')
print(transition.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(transition, annot=True, fmt='.2f', cmap='Blues', ax=ax)
ax.set_title('Job Family → Tenure Transition Probabilities')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/05_transition_matrix.png', bbox_inches='tight')
plt.show()

## 3. K-Means Clustering (Career Archetypes)

In [ ]:
cluster_features = ['tenure_years', 'seniority_level']
cat_cluster_cols = ['job_family', 'DepartmentType']
for col in cat_cluster_cols:
    le = LabelEncoder()
    df[col] = df[col].fillna('Unknown').astype(str)
    df[f'{col}_enc'] = le.fit_transform(df[col])
    cluster_features.append(f'{col}_enc')

X_cluster = df[cluster_features].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Find optimal k using elbow
inertias = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(2, 9), inertias, marker='o')
ax.set_title('Elbow Method for Optimal K')
ax.set_xlabel('Number of Clusters')
ax.set_ylabel('Inertia')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/05_elbow_k.png', bbox_inches='tight')
plt.show()

In [ ]:
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
df['career_cluster'] = kmeans.fit_predict(X_scaled)

# Profile clusters
cluster_profile = df.groupby('career_cluster').agg({
    'tenure_years': 'mean',
    'seniority_level': 'mean',
    'EmpID': 'count',
}).round(2)
cluster_profile.columns = ['Avg Tenure', 'Avg Seniority', 'Count']
print('Career Archetypes:')
print(cluster_profile)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
pd.crosstab(df['career_cluster'], df['job_family'], normalize='index').plot(
    kind='bar', stacked=True, ax=ax, title=f'Job Family Distribution by Cluster')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/05_cluster_job_family.png', bbox_inches='tight')
plt.show()

## 4. Key Takeaways

In [ ]:
print('--- Key Insights ---')
print(f'1. Identified {optimal_k} career archetypes')
print(f'2. Average tenure: {df["tenure_years"].mean():.1f} years')
print()
print('--- HR Action Items ---')
print('- Develop mobility programs targeting low-tenure clusters')
print('- Create career pathways across job families')
print()
print('--- Employee Impact ---')
print('- Clear career path visibility for all employees')
print('- Clustering reveals hidden mobility opportunities')